# 🔬 Demonstração de Clusterização: K-Means vs DBSCAN

Este notebook demonstra e compara dois algoritmos de clusterização amplamente utilizados:
- **K-Means**: Clusterização baseada em centroides, requer definição prévia do número de clusters *k*
- **DBSCAN**: Clusterização baseada em densidade, detecta automaticamente o número de clusters e lida bem com ruídos/outliers

---
### 📋 Fluxo do Notebook
1. Instalação e importação das bibliotecas
2. Geração / carregamento dos dados
3. Pré-processamento
4. **K-Means** — seleção de *k* via Curva do Cotovelo e Índice de Silhueta
5. **DBSCAN** — ajuste de parâmetros
6. Comparação dos resultados


## 1. 📦 Instalação e Importação de Bibliotecas

In [1]:
# Instalar dependências (caso necessário no Colab)
!pip install plotly scikit-learn pandas numpy -q

In [2]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score, calinski_harabasz_score

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')

# Semente para reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ Bibliotecas importadas com sucesso!')

✅ Bibliotecas importadas com sucesso!


## 2. 📊 Dados

> **ℹ️ Dados Dummy Ativos**  
> O notebook está usando dados sintéticos gerados automaticamente.  
> Quando você quiser usar seus dados reais, substitua o conteúdo da célula marcada como **`[SUBSTITUIR DADOS REAIS]`** abaixo.


In [ ]:
# ==============================================================
# CONFIGURAÇÃO: Escolha o tipo de dado dummy para demonstração
# ==============================================================
# Opções:
#   'blobs'      → clusters esféricos e bem separados (favorece K-Means)
#   'moons'      → clusters em formato de meia-lua (favorece DBSCAN)
#   'mixed'      → combinação de formatos variados (desafiador para ambos)

DATASET_TYPE = 'mixed'   # << Altere aqui para testar cenários diferentes
N_SAMPLES    = 600
N_FEATURES   = 2         # 2 para visualização 2D, ou mais para dados reais

# --------------------------------------------------------------
def gerar_dados_dummy(tipo: str, n_samples: int, n_features: int) -> pd.DataFrame:
    """Gera diferentes tipos de dados sintéticos para demonstração."""

    if tipo == 'blobs':
        X, _ = make_blobs(
            n_samples=n_samples,
            n_features=n_features,
            centers=4,
            cluster_std=[1.0, 1.5, 0.8, 1.2],
            random_state=RANDOM_STATE
        )

    elif tipo == 'moons':
        X, _ = make_moons(n_samples=n_samples, noise=0.08, random_state=RANDOM_STATE)

    elif tipo == 'mixed':
        # Blobs
        X1, _ = make_blobs(n_samples=int(n_samples * 0.5), centers=[
            [-6, -6], [6, 6]
        ], cluster_std=0.8, random_state=RANDOM_STATE)
        # Moons
        X2, _ = make_moons(n_samples=int(n_samples * 0.35), noise=0.07, random_state=RANDOM_STATE)
        X2 = X2 * 2 + np.array([0, 3])
        # Outliers
        X3 = np.random.uniform(-10, 10, size=(int(n_samples * 0.05), 2))

        X = np.vstack([X1, X2, X3])
    else:
        raise ValueError(f"Tipo desconhecido: {tipo}. Use 'blobs', 'moons' ou 'mixed'.")

    cols = [f'feature_{i+1}' for i in range(X.shape[1])]
    return pd.DataFrame(X, columns=cols)

# --------------------------------------------------------------
df = gerar_dados_dummy(DATASET_TYPE, N_SAMPLES, N_FEATURES)

print(f'✅ Dataset gerado: tipo="{DATASET_TYPE}" | shape={df.shape}')
df.head()

In [11]:
from google.colab import files
uploaded = files.upload()
import io
filename = list(uploaded.keys())[0]
df = pd.read_excel(io.BytesIO(uploaded[filename]), skiprows=2)[:-6]
df

Saving Dispersao_Scores_DRADS.xlsx to Dispersao_Scores_DRADS.xlsx


,DRADS,X – Capacidade de Execução,Y – Proporção de Esforço Socioassistencial,Quadrante
0,Alta Noroeste,2.5,3.33,Q2 – Alta prio. / Alta cap.
1,Alta Paulista,1.75,2.24,Q2 – Alta prio. / Alta cap.
2,Alta Sorocaba,1.5,2.09,Q2 – Alta prio. / Alta cap.
3,Avaré,8,5.99,Q2 – Alta prio. / Alta cap.
4,Baixada Santista,-0.5,-0.01,Q3 – Baixa prio. / Baixa cap.
5,Barretos,3.5,0.49,Q2 – Alta prio. / Alta cap.
6,Bauru,-1,-0.99,Q3 – Baixa prio. / Baixa cap.
7,Botucatu,2,1.38,Q2 – Alta prio. / Alta cap.
8,Campinas,-1,0.27,Q1 – Alta prio. / Baixa cap.
9,Capital,2.25,2.21,Q2 – Alta prio. / Alta cap.


In [50]:
#Defina as colunas que serão usadas na clusterização:
FEATURE_COLS = ['X – Capacidade de Execução', 'Y – Proporção de Esforço Socioassistencial']  # Deixe None para usar todas
PADRONIZAR = False

## 3. 🔧 Pré-processamento

In [15]:
# Selecionar features numéricas
if FEATURE_COLS is None:
    FEATURE_COLS = df.select_dtypes(include=[np.number]).columns.tolist()

X_raw = df[FEATURE_COLS].dropna()

# Padronização (z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f'Features utilizadas : {FEATURE_COLS}')
print(f'Shape após limpeza  : {X_raw.shape}')
print(f'Shape padronizado   : {X_scaled.shape}')

# Estatísticas descritivas
df[FEATURE_COLS].describe().round(3)

Features utilizadas : ['X – Capacidade de Execução', 'Y – Proporção de Esforço Socioassistencial']
Shape após limpeza  : (25, 2)
Shape padronizado   : (25, 2)


,Y – Proporção de Esforço Socioassistencial
count,25.000
mean,1.483
std,1.815
min,-1.000
25%,-0.080
50%,1.560
75%,2.780
max,5.990


In [51]:
# Visualização da distribuição original dos dados
# (Apenas para datasets 2D; para N>2 usa pairplot)

if PADRONIZAR:
  df_plot = pd.DataFrame(X_scaled, columns=FEATURE_COLS)
else:
  df_plot = pd.DataFrame(X_raw, columns=FEATURE_COLS)

if len(FEATURE_COLS) == 2:
    fig = px.scatter(
        df_plot,
        x=FEATURE_COLS[0],
        y=FEATURE_COLS[1],
        title='Distribuição Original dos Dados (padronizados)',
        template='plotly_white',
        color_discrete_sequence=['#636EFA'],
        opacity=0.7
    )
    fig.update_traces(marker=dict(size=6))
    fig.update_layout(height=500)
    fig.show()

elif len(FEATURE_COLS) >= 3:
    fig = px.scatter_matrix(
        df_plot,
        dimensions=FEATURE_COLS[:6],  # limita a 6 para legibilidade
        title='Matriz de Dispersão — Dados Padronizados',
        template='plotly_white',
        opacity=0.5
    )
    fig.update_layout(height=700)
    fig.show()

## 4. 🎯 K-Means
### 4.1 Seleção do Número de Clusters (*k*)

In [52]:
# --------------------------------------------------------------
# CONFIGURAÇÃO K-Means
# --------------------------------------------------------------
K_RANGE    = range(2, 12)   # Faixa de k a avaliar
N_INIT     = 10             # Inicializações aleatórias por k
MAX_ITER   = 300
if PADRONIZAR:
  X = X_scaled
else:
  X = X_raw

inertias          = []
silhouette_scores = []
db_scores         = []
ch_scores         = []

print('Avaliando valores de k...', end=' ')
for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=N_INIT,
                max_iter=MAX_ITER, random_state=RANDOM_STATE)
    labels = km.fit_predict(X)

    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X, labels))
    db_scores.append(davies_bouldin_score(X, labels))
    ch_scores.append(calinski_harabasz_score(X, labels))

print('✅')

# Sugestão automática de k ótimo (maior índice de silhueta)
best_k_silhouette = list(K_RANGE)[np.argmax(silhouette_scores)]
best_k_ch         = list(K_RANGE)[np.argmax(ch_scores)]

print(f'\n🏆 k sugerido pelo Índice de Silhueta     : {best_k_silhouette}')
print(f'🏆 k sugerido pelo Índice Calinski-Harabasz: {best_k_ch}')

Avaliando valores de k... ✅

🏆 k sugerido pelo Índice de Silhueta     : 8
🏆 k sugerido pelo Índice Calinski-Harabasz: 10


In [53]:
# Curva do Cotovelo + Índice de Silhueta — subplots lado a lado

k_list = list(K_RANGE)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Curva do Cotovelo (Inércia)', 'Índice de Silhueta')
)

# --- Cotovelo ---
fig.add_trace(
    go.Scatter(x=k_list, y=inertias, mode='lines+markers',
               name='Inércia',
               line=dict(color='#636EFA', width=2.5),
               marker=dict(size=8, symbol='circle')),
    row=1, col=1
)

# --- Silhueta ---
colors_sil = ['#EF553B' if k == best_k_silhouette else '#00CC96' for k in k_list]
fig.add_trace(
    go.Bar(x=k_list, y=silhouette_scores, name='Silhueta',
           marker_color=colors_sil,
           text=[f'{v:.3f}' for v in silhouette_scores],
           textposition='outside'),
    row=1, col=2
)
fig.add_vline(x=best_k_silhouette, line_dash='dash', line_color='#EF553B',
              annotation_text=f' k={best_k_silhouette} (melhor)',
              annotation_position='top right', row=1, col=2)

fig.update_layout(
    title_text='Seleção do k Ótimo para K-Means',
    template='plotly_white',
    height=450,
    showlegend=False
)
fig.update_xaxes(title_text='Número de Clusters (k)', dtick=1)
fig.update_yaxes(title_text='Inércia (WCSS)', row=1, col=1)
fig.update_yaxes(title_text='Silhouette Score', row=1, col=2, range=[0, 1])

fig.show()

In [54]:
# Índices complementares: Davies-Bouldin e Calinski-Harabasz

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Davies-Bouldin (↓ melhor)', 'Calinski-Harabasz (↑ melhor)')
)

fig.add_trace(
    go.Scatter(x=k_list, y=db_scores, mode='lines+markers',
               name='Davies-Bouldin',
               line=dict(color='#AB63FA', width=2.5),
               marker=dict(size=8)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=k_list, y=ch_scores, mode='lines+markers',
               name='Calinski-Harabasz',
               line=dict(color='#FFA15A', width=2.5),
               marker=dict(size=8)),
    row=1, col=2
)

fig.update_layout(
    title_text='Índices de Avaliação de Clustering — K-Means',
    template='plotly_white',
    height=420,
    showlegend=False
)
fig.update_xaxes(title_text='Número de Clusters (k)', dtick=1)
fig.show()

### 4.2 Aplicação do K-Means com o *k* Selecionado

In [55]:
# ==============================================================
# Defina o k final aqui (ou mantenha a sugestão automática)
# ==============================================================
K_FINAL = best_k_silhouette  # << Altere se preferir outro valor

km_final = KMeans(n_clusters=K_FINAL, init='k-means++', n_init=N_INIT,
                  max_iter=MAX_ITER, random_state=RANDOM_STATE)
km_labels = km_final.fit_predict(X)

centroids_scaled = km_final.cluster_centers_

sil_final = silhouette_score(X, km_labels)
print(f'K-Means | k={K_FINAL} | Silhouette={sil_final:.4f} | Inércia={km_final.inertia_:.2f}')

K-Means | k=8 | Silhouette=0.4648 | Inércia=7.51


In [56]:
# Visualização dos clusters K-Means (2D)

df_km = pd.DataFrame(X, columns=FEATURE_COLS)
df_km['Cluster'] = km_labels.astype(str)

centroids_df = pd.DataFrame(centroids_scaled, columns=FEATURE_COLS)

if len(FEATURE_COLS) >= 2:
    xcol, ycol = FEATURE_COLS[0], FEATURE_COLS[1]

    fig = px.scatter(
        df_km, x=xcol, y=ycol,
        color='Cluster',
        title=f'K-Means — k={K_FINAL} | Silhouette={sil_final:.4f}',
        template='plotly_white',
        opacity=0.75,
        color_discrete_sequence=px.colors.qualitative.Bold
    )
    fig.update_traces(marker=dict(size=7))

    # Adicionar centroides
    fig.add_trace(go.Scatter(
        x=centroids_df[xcol],
        y=centroids_df[ycol],
        mode='markers',
        marker=dict(symbol='x', size=16, color='black',
                    line=dict(width=2, color='white')),
        name='Centroides'
    ))

    fig.update_layout(height=550, legend_title_text='Cluster')
    fig.show()

In [57]:
# Diagrama de Silhueta por Cluster

sil_samples = silhouette_samples(X, km_labels)

fig = go.Figure()
y_lower = 0
colors = px.colors.qualitative.Bold

for i in range(K_FINAL):
    sil_i = np.sort(sil_samples[km_labels == i])
    size_i = len(sil_i)
    y_upper = y_lower + size_i

    fig.add_trace(go.Bar(
        x=sil_i,
        y=list(range(y_lower, y_upper)),
        orientation='h',
        name=f'Cluster {i}',
        marker_color=colors[i % len(colors)],
        showlegend=True,
        width=1
    ))
    y_lower = y_upper + 10

fig.add_vline(x=sil_final, line_dash='dash', line_color='red',
              annotation_text=f'Média={sil_final:.3f}',
              annotation_position='top right')

fig.update_layout(
    title=f'Diagrama de Silhueta — K-Means (k={K_FINAL})',
    xaxis_title='Coeficiente de Silhueta',
    yaxis_title='Amostras por Cluster',
    template='plotly_white',
    height=500,
    yaxis=dict(showticklabels=False),
    bargap=0
)
fig.show()

## 5. 🌀 DBSCAN
### 5.1 Ajuste de Parâmetros via k-Distance Plot

In [71]:
from sklearn.neighbors import NearestNeighbors

# ==============================================================
# CONFIGURAÇÃO DBSCAN
# ==============================================================
DBSCAN_EPS         = 1.2     # << Raio de vizinhança  (ajuste se necessário)
DBSCAN_MIN_SAMPLES = 2      # << Mínimo de pontos para formar núcleo

# k-Distance Plot para auxiliar na escolha do eps
k_neighbors = DBSCAN_MIN_SAMPLES
nbrs = NearestNeighbors(n_neighbors=k_neighbors).fit(X)
distances, _ = nbrs.kneighbors(X)
k_dist = np.sort(distances[:, -1])[::-1]

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=k_dist, mode='lines',
    line=dict(color='#19D3F3', width=2),
    name='k-distância'
))
fig.add_hline(y=DBSCAN_EPS, line_dash='dash', line_color='#EF553B',
              annotation_text=f' eps={DBSCAN_EPS}',
              annotation_position='right')

fig.update_layout(
    title=f'k-Distance Plot (k={k_neighbors}) — Auxiliar para eps do DBSCAN',
    xaxis_title='Índice da Amostra (ordenado)',
    yaxis_title=f'Distância ao {k_neighbors}º Vizinho',
    template='plotly_white',
    height=430
)
fig.show()
print('💡 Dica: O eps ideal costuma estar no ponto de maior curvatura ("cotovelo") do gráfico.')

💡 Dica: O eps ideal costuma estar no ponto de maior curvatura ("cotovelo") do gráfico.


### 5.2 Aplicação do DBSCAN

In [72]:
db = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric='euclidean')
db_labels = db.fit_predict(X)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise_db    = np.sum(db_labels == -1)
noise_pct     = n_noise_db / len(db_labels) * 100

# Silhouette apenas se houver mais de 1 cluster e sem todos os pontos como ruído
if n_clusters_db > 1:
    mask_valid = db_labels != -1
    if mask_valid.sum() > 1:
        sil_db = silhouette_score(X[mask_valid], db_labels[mask_valid])
    else:
        sil_db = np.nan
else:
    sil_db = np.nan

print(f'DBSCAN | eps={DBSCAN_EPS} | min_samples={DBSCAN_MIN_SAMPLES}')
print(f'       | Clusters encontrados : {n_clusters_db}')
print(f'       | Ruídos (outliers)    : {n_noise_db} ({noise_pct:.1f}%)')
print(f'       | Silhouette (sem ruído): {sil_db:.4f}' if not np.isnan(sil_db) else '       | Silhouette: N/A')

DBSCAN | eps=1.2 | min_samples=2
       | Clusters encontrados : 4
       | Ruídos (outliers)    : 2 (8.0%)
       | Silhouette (sem ruído): 0.4934


In [73]:
# Visualização dos clusters DBSCAN

df_db = pd.DataFrame(X, columns=FEATURE_COLS)
df_db['Cluster'] = db_labels
df_db['Tipo'] = df_db['Cluster'].apply(lambda x: 'Ruído (−1)' if x == -1 else f'Cluster {x}')

if len(FEATURE_COLS) >= 2:
    xcol, ycol = FEATURE_COLS[0], FEATURE_COLS[1]

    color_map = {}
    palette = px.colors.qualitative.Bold
    unique_types = sorted(df_db['Tipo'].unique(), key=lambda x: (x == 'Ruído (−1)', x))
    for i, t in enumerate(unique_types):
        color_map[t] = '#888888' if t == 'Ruído (−1)' else palette[i % len(palette)]

    fig = px.scatter(
        df_db, x=xcol, y=ycol,
        color='Tipo',
        color_discrete_map=color_map,
        title=f'DBSCAN — eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES} | '
              f'Clusters={n_clusters_db} | Ruídos={n_noise_db} ({noise_pct:.1f}%)',
        template='plotly_white',
        opacity=0.75,
        symbol='Tipo',
        symbol_map={'Ruído (−1)': 'x'}
    )
    fig.update_traces(marker=dict(size=7))
    fig.update_layout(height=550, legend_title_text='Rótulo')
    fig.show()

In [80]:
# Visualização dos clusters DBSCAN com zonas de calor

df_db = pd.DataFrame(X, columns=FEATURE_COLS)
df_db['Cluster'] = db_labels
df_db['Tipo'] = df_db['Cluster'].apply(lambda x: 'Ruído (−1)' if x == -1 else f'Cluster {x}')

if len(FEATURE_COLS) >= 2:
    xcol, ycol = FEATURE_COLS[0], FEATURE_COLS[1]

    palette = px.colors.qualitative.Bold
    unique_types = sorted(df_db['Tipo'].unique(), key=lambda x: (x == 'Ruído (−1)', x))
    color_map = {}
    for i, t in enumerate(unique_types):
        color_map[t] = '#888888' if t == 'Ruído (−1)' else palette[i % len(palette)]

    # --- Grade para zonas de calor ---
    x_vals = X_scaled[:, 0]
    y_vals = X_scaled[:, 1]
    margin = DBSCAN_EPS * 2
    x_min, x_max = x_vals.min() - margin, x_vals.max() + margin
    y_min, y_max = y_vals.min() - margin, y_vals.max() + margin

    grid_res = 300
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, grid_res),
        np.linspace(y_min, y_max, grid_res)
    )
    grid_points = np.c_[xx.ravel(), yy.ravel()]

    # Para cada ponto da grade, encontra o cluster DBSCAN mais próximo
    # usando apenas os pontos núcleo (não-ruído) como referência
    from sklearn.neighbors import NearestNeighbors

    core_mask   = db_labels != -1
    core_points = X[core_mask]
    core_labels = db_labels[core_mask]

    # Mapeia cada célula da grade ao cluster do vizinho mais próximo (entre pontos núcleo)
    if len(core_points) > 0:
        nbrs = NearestNeighbors(n_neighbors=1).fit(core_points)
        dists, idxs = nbrs.kneighbors(grid_points)
        nearest_label  = core_labels[idxs.ravel()]
        nearest_dist   = dists.ravel()

        # Células além do raio eps ficam como "sem cluster" (−1)
        grid_labels = np.where(nearest_dist <= DBSCAN_EPS, nearest_label, -1)
    else:
        grid_labels = np.full(len(grid_points), -1)

    zz = grid_labels.reshape(xx.shape)

    # Paleta de cores para o heatmap: mapeia ids de cluster → índice de cor
    unique_clusters = sorted(set(db_labels))
    cluster_to_idx  = {c: i for i, c in enumerate(unique_clusters)}

    zz_idx = np.vectorize(lambda c: cluster_to_idx.get(c, 0))(zz)

    # Constrói colorscale discreta alinhada com a paleta dos pontos
    n_levels = len(unique_clusters)
    colorscale = []
    for i, c in enumerate(unique_clusters):
        color = '#E8E8E8' if c == -1 else palette[i % len(palette)]
        colorscale.append([i / n_levels,       color])
        colorscale.append([(i + 1) / n_levels, color])

    fig = go.Figure()

    # Camada 1 — zonas de calor (heatmap de fundo)
    fig.add_trace(go.Heatmap(
        x=np.linspace(x_min, x_max, grid_res),
        y=np.linspace(y_min, y_max, grid_res),
        z=zz_idx,
        colorscale=colorscale,
        zmin=0,
        zmax=n_levels,
        showscale=False,
        opacity=0.25,
        hoverinfo='skip',
        name='Zona de calor'
    ))

    # Camada 2 — contorno das zonas (isolinhas entre clusters)
    fig.add_trace(go.Contour(
        x=np.linspace(x_min, x_max, grid_res),
        y=np.linspace(y_min, y_max, grid_res),
        z=zz_idx,
        showscale=False,
        colorscale=colorscale,
        contours=dict(
            start=0, end=n_levels, size=1,
            coloring='none'
        ),
        line=dict(width=1.2, color='rgba(80,80,80,0.35)', dash='dot'),
        hoverinfo='skip',
        name='Fronteira'
    ))

    # Camada 3 — pontos dos clusters
    for i, t in enumerate(unique_types):
        mask = df_db['Tipo'] == t
        is_noise = t == 'Ruído (−1)'
        fig.add_trace(go.Scatter(
            x=df_db.loc[mask, xcol],
            y=df_db.loc[mask, ycol],
            mode='markers',
            name=t,
            marker=dict(
                color=color_map[t],
                size=5 if is_noise else 7,
                symbol='x' if is_noise else 'circle',
                opacity=0.55 if is_noise else 0.85,
                line=dict(width=0.5, color='white') if not is_noise else dict(width=1)
            )
        ))

    fig.update_layout(
        title=f'DBSCAN — eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES} | '
              f'Clusters={n_clusters_db} | Ruídos={n_noise_db} ({noise_pct:.1f}%)',
        xaxis_title=xcol,
        yaxis_title=ycol,
        template='plotly_white',
        height=580,
        legend_title_text='Rótulo',
        xaxis=dict(range=[x_min, x_max]),
        yaxis=dict(range=[y_min, y_max])
    )
    fig.show()

In [83]:
# Visualização dos clusters DBSCAN com zonas de calor (dados não normalizados + labels DRADS)

df_db = pd.DataFrame(X_raw, columns=FEATURE_COLS).reset_index(drop=True)
df_db['Cluster'] = db_labels
df_db['Tipo'] = df_db['Cluster'].apply(lambda x: 'Ruído (−1)' if x == -1 else f'Cluster {x}')
df_db['DRADS'] = df['DRADS'].reset_index(drop=True)

if len(FEATURE_COLS) >= 2:
    xcol, ycol = FEATURE_COLS[0], FEATURE_COLS[1]

    X_raw_np = X_raw.to_numpy()  # <- conversão para numpy aqui

    palette = px.colors.qualitative.Bold
    unique_types = sorted(df_db['Tipo'].unique(), key=lambda x: (x == 'Ruído (−1)', x))
    color_map = {}
    for i, t in enumerate(unique_types):
        color_map[t] = '#888888' if t == 'Ruído (−1)' else palette[i % len(palette)]

    # --- Grade para zonas de calor (espaço original, não normalizado) ---
    x_vals = X_raw_np[:, 0]
    y_vals = X_raw_np[:, 1]
    margin_x = (x_vals.max() - x_vals.min()) * 0.05
    margin_y = (y_vals.max() - y_vals.min()) * 0.05
    x_min, x_max = x_vals.min() - margin_x, x_vals.max() + margin_x
    y_min, y_max = y_vals.min() - margin_y, y_vals.max() + margin_y

    grid_res = 300
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, grid_res),
        np.linspace(y_min, y_max, grid_res)
    )
    grid_points = np.c_[xx.ravel(), yy.ravel()]

    from sklearn.neighbors import NearestNeighbors

    core_mask   = db_labels != -1
    core_points = X_raw_np[core_mask]
    core_labels = db_labels[core_mask]

    if len(core_points) > 0:
        nbrs = NearestNeighbors(n_neighbors=1).fit(core_points)
        dists, idxs = nbrs.kneighbors(grid_points)
        nearest_label = core_labels[idxs.ravel()]
        nearest_dist  = dists.ravel()

        eps_original = DBSCAN_EPS * np.mean(scaler.scale_)
        grid_labels  = np.where(nearest_dist <= eps_original, nearest_label, -1)
    else:
        grid_labels = np.full(len(grid_points), -1)

    zz = grid_labels.reshape(xx.shape)

    unique_clusters = sorted(set(db_labels))
    cluster_to_idx  = {c: i for i, c in enumerate(unique_clusters)}
    zz_idx = np.vectorize(lambda c: cluster_to_idx.get(c, 0))(zz)

    n_levels = len(unique_clusters)
    colorscale = []
    for i, c in enumerate(unique_clusters):
        color = '#E8E8E8' if c == -1 else palette[i % len(palette)]
        colorscale.append([i / n_levels,       color])
        colorscale.append([(i + 1) / n_levels, color])

    fig = go.Figure()

    # Camada 1 — zonas de calor
    fig.add_trace(go.Heatmap(
        x=np.linspace(x_min, x_max, grid_res),
        y=np.linspace(y_min, y_max, grid_res),
        z=zz_idx,
        colorscale=colorscale,
        zmin=0,
        zmax=n_levels,
        showscale=False,
        opacity=0.25,
        hoverinfo='skip',
        name='Zona de calor'
    ))

    # Camada 2 — contorno das zonas
    fig.add_trace(go.Contour(
        x=np.linspace(x_min, x_max, grid_res),
        y=np.linspace(y_min, y_max, grid_res),
        z=zz_idx,
        showscale=False,
        colorscale=colorscale,
        contours=dict(start=0, end=n_levels, size=1, coloring='none'),
        line=dict(width=1.2, color='rgba(80,80,80,0.35)', dash='dot'),
        hoverinfo='skip',
        name='Fronteira'
    ))

# Camada 3 — pontos com labels DRADS
    for i, t in enumerate(unique_types):
        mask = df_db['Tipo'] == t
        is_noise = t == 'Ruído (−1)'
        subset = df_db[mask]

        fig.add_trace(go.Scatter(
            x=subset[xcol],
            y=subset[ycol],
            mode='markers+text',
            name=t,
            text=subset['DRADS'],
            textposition='top center',
            textfont=dict(
                size=9,
                color='#444444' if not is_noise else '#AAAAAA'
            ),
            marker=dict(
                color=color_map[t],
                size=5 if is_noise else 7,
                symbol='x' if is_noise else 'circle',
                opacity=0.55 if is_noise else 0.85,
                line=dict(width=0.5, color='white') if not is_noise else dict(width=1)
            ),
            cliponaxis=False,   # <- pontos nunca são cortados pelo eixo
            hovertemplate=(
                '<b>%{text}</b><br>'
                f'{xcol}: %{{x:.3f}}<br>'
                f'{ycol}: %{{y:.3f}}<br>'
                f'Cluster: {t}<extra></extra>'
            )
        ))

    fig.update_layout(
        title=f'DBSCAN — eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES} | '
              f'Clusters={n_clusters_db} | Ruídos={n_noise_db} ({noise_pct:.1f}%)',
        xaxis_title=xcol,
        yaxis_title=ycol,
        template='plotly_white',
        height=580,
        legend_title_text='Rótulo',
        xaxis=dict(range=[x_min, x_max]),  # mantém o zoom inicial no heatmap
        yaxis=dict(range=[y_min, y_max])   # mas os pontos não são cortados
    )
    fig.show()

In [85]:
from scipy.spatial import ConvexHull

fig = go.Figure()

# Camada 1 — zonas de calor via ConvexHull (expande com o zoom)
for i, c in enumerate(unique_clusters):
    if c == -1:
        continue
    pts = X_raw_np[db_labels == c]
    if len(pts) < 3:
        continue
    try:
        hull = ConvexHull(pts)
        hull_pts = pts[hull.vertices]
        # Fecha o polígono repetindo o primeiro ponto
        hx = np.append(hull_pts[:, 0], hull_pts[0, 0])
        hy = np.append(hull_pts[:, 1], hull_pts[0, 1])
    except Exception:
        continue

    color_rgb = palette[i % len(palette)]
    fig.add_trace(go.Scatter(
        x=hx, y=hy,
        mode='lines',
        fill='toself',
        fillcolor=color_rgb,
        opacity=0.15,
        line=dict(color=color_rgb, width=1.5, dash='dot'),
        showlegend=False,
        hoverinfo='skip',
        name=f'Zona Cluster {c}'
    ))

# Camada 2 — pontos com labels DRADS
for i, t in enumerate(unique_types):
    mask = df_db['Tipo'] == t
    is_noise = t == 'Ruído (−1)'
    subset = df_db[mask]

    fig.add_trace(go.Scatter(
        x=subset[xcol],
        y=subset[ycol],
        mode='markers+text',
        name=t,
        text=subset['DRADS'],
        textposition='top center',
        textfont=dict(
            size=9,
            color='#444444' if not is_noise else '#AAAAAA'
        ),
        marker=dict(
            color=color_map[t],
            size=5 if is_noise else 7,
            symbol='x' if is_noise else 'circle',
            opacity=0.55 if is_noise else 0.85,
            line=dict(width=0.5, color='white') if not is_noise else dict(width=1)
        ),
        cliponaxis=False,
        hovertemplate=(
            '<b>%{text}</b><br>'
            f'{xcol}: %{{x:.3f}}<br>'
            f'{ycol}: %{{y:.3f}}<br>'
            f'Cluster: {t}<extra></extra>'
        )
    ))

fig.update_layout(
    title=f'DBSCAN — eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES} | '
          f'Clusters={n_clusters_db} | Ruídos={n_noise_db} ({noise_pct:.1f}%)',
    xaxis_title=xcol,
    yaxis_title=ycol,
    template='plotly_white',
    height=580,
    legend_title_text='Rótulo',
)
fig.show()

In [86]:
# Visualização dos clusters DBSCAN com zonas de calor (dados não normalizados + labels DRADS)

df_db = pd.DataFrame(X_raw, columns=FEATURE_COLS).reset_index(drop=True)
df_db['Cluster'] = db_labels
df_db['Tipo'] = df_db['Cluster'].apply(lambda x: 'Ruído (−1)' if x == -1 else f'Cluster {x}')
df_db['DRADS'] = df['DRADS'].reset_index(drop=True)

if len(FEATURE_COLS) >= 2:
    xcol, ycol = FEATURE_COLS[0], FEATURE_COLS[1]

    X_raw_np = X_raw.to_numpy()

    palette = px.colors.qualitative.Bold
    unique_types = sorted(df_db['Tipo'].unique(), key=lambda x: (x == 'Ruído (−1)', x))
    color_map = {}
    for i, t in enumerate(unique_types):
        color_map[t] = '#888888' if t == 'Ruído (−1)' else palette[i % len(palette)]

    # --- Grade para zonas de calor (grade expandida para suportar zoom out) ---
    x_vals = X_raw_np[:, 0]
    y_vals = X_raw_np[:, 1]

    span_x = x_vals.max() - x_vals.min()
    span_y = y_vals.max() - y_vals.min()

    margin_x = span_x * 5
    margin_y = span_y * 5

    x_min, x_max = x_vals.min() - margin_x, x_vals.max() + margin_x
    y_min, y_max = y_vals.min() - margin_y, y_vals.max() + margin_y

    view_margin_x = span_x * 0.10
    view_margin_y = span_y * 0.10
    x_view_min, x_view_max = x_vals.min() - view_margin_x, x_vals.max() + view_margin_x
    y_view_min, y_view_max = y_vals.min() - view_margin_y, y_vals.max() + view_margin_y

    grid_res = 300
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, grid_res),
        np.linspace(y_min, y_max, grid_res)
    )
    grid_points = np.c_[xx.ravel(), yy.ravel()]

    from sklearn.neighbors import NearestNeighbors

    core_mask   = db_labels != -1
    core_points = X_raw_np[core_mask]
    core_labels = db_labels[core_mask]

    if len(core_points) > 0:
        nbrs = NearestNeighbors(n_neighbors=1).fit(core_points)
        dists, idxs = nbrs.kneighbors(grid_points)
        nearest_label = core_labels[idxs.ravel()]
        nearest_dist  = dists.ravel()

        eps_original = DBSCAN_EPS * np.mean(scaler.scale_)
        grid_labels  = np.where(nearest_dist <= eps_original, nearest_label, -1)
    else:
        grid_labels = np.full(len(grid_points), -1)

    zz = grid_labels.reshape(xx.shape)

    unique_clusters = sorted(set(db_labels))
    cluster_to_idx  = {c: i for i, c in enumerate(unique_clusters)}
    zz_idx = np.vectorize(lambda c: cluster_to_idx.get(c, 0))(zz)

    n_levels = len(unique_clusters)
    colorscale = []
    for i, c in enumerate(unique_clusters):
        color = '#E8E8E8' if c == -1 else palette[i % len(palette)]
        colorscale.append([i / n_levels,       color])
        colorscale.append([(i + 1) / n_levels, color])

    fig = go.Figure()

    # Camada 1 — zonas de calor
    fig.add_trace(go.Heatmap(
        x=np.linspace(x_min, x_max, grid_res),
        y=np.linspace(y_min, y_max, grid_res),
        z=zz_idx,
        colorscale=colorscale,
        zmin=0,
        zmax=n_levels,
        showscale=False,
        opacity=0.25,
        hoverinfo='skip',
        name='Zona de calor'
    ))

    # Camada 2 — contorno das zonas
    fig.add_trace(go.Contour(
        x=np.linspace(x_min, x_max, grid_res),
        y=np.linspace(y_min, y_max, grid_res),
        z=zz_idx,
        showscale=False,
        colorscale=colorscale,
        contours=dict(start=0, end=n_levels, size=1, coloring='none'),
        line=dict(width=1.2, color='rgba(80,80,80,0.35)', dash='dot'),
        hoverinfo='skip',
        name='Fronteira'
    ))

    # Camada 3 — pontos com labels DRADS
    for i, t in enumerate(unique_types):
        mask = df_db['Tipo'] == t
        is_noise = t == 'Ruído (−1)'
        subset = df_db[mask]

        fig.add_trace(go.Scatter(
            x=subset[xcol],
            y=subset[ycol],
            mode='markers+text',
            name=t,
            text=subset['DRADS'],
            textposition='top center',
            textfont=dict(
                size=9,
                color='#444444' if not is_noise else '#AAAAAA'
            ),
            marker=dict(
                color=color_map[t],
                size=5 if is_noise else 7,
                symbol='x' if is_noise else 'circle',
                opacity=0.55 if is_noise else 0.85,
                line=dict(width=0.5, color='white') if not is_noise else dict(width=1)
            ),
            cliponaxis=False,
            hovertemplate=(
                '<b>%{text}</b><br>'
                f'{xcol}: %{{x:.3f}}<br>'
                f'{ycol}: %{{y:.3f}}<br>'
                f'Cluster: {t}<extra></extra>'
            )
        ))

    fig.update_layout(
        title=f'DBSCAN — eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES} | '
              f'Clusters={n_clusters_db} | Ruídos={n_noise_db} ({noise_pct:.1f}%)',
        xaxis_title=xcol,
        yaxis_title=ycol,
        template='plotly_white',
        height=580,
        legend_title_text='Rótulo',
        xaxis=dict(range=[x_view_min, x_view_max]),
        yaxis=dict(range=[y_view_min, y_view_max])
    )
    fig.show()

## 6. ⚖️ Comparação: K-Means vs DBSCAN

In [45]:
# Painel comparativo lado a lado dos clusters

if len(FEATURE_COLS) >= 2:
    xcol, ycol = FEATURE_COLS[0], FEATURE_COLS[1]

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'K-Means (k={K_FINAL})',
            f'DBSCAN (eps={DBSCAN_EPS}, min_s={DBSCAN_MIN_SAMPLES})'
        )
    )

    palette = px.colors.qualitative.Bold

    # K-Means
    for c in sorted(set(km_labels)):
        mask = km_labels == c
        fig.add_trace(
            go.Scatter(
                x=X_scaled[mask, 0], y=X_scaled[mask, 1],
                mode='markers', name=f'KM-{c}',
                marker=dict(color=palette[c % len(palette)], size=6, opacity=0.7)
            ), row=1, col=1
        )
    # Centroides
    fig.add_trace(
        go.Scatter(
            x=centroids_scaled[:, 0], y=centroids_scaled[:, 1],
            mode='markers', name='Centroides',
            marker=dict(symbol='x', size=14, color='black', line=dict(width=2))
        ), row=1, col=1
    )

    # DBSCAN
    for i, c in enumerate(sorted(set(db_labels))):
        mask = db_labels == c
        is_noise = c == -1
        fig.add_trace(
            go.Scatter(
                x=X_scaled[mask, 0], y=X_scaled[mask, 1],
                mode='markers',
                name='Ruído' if is_noise else f'DB-{c}',
                marker=dict(
                    color='#AAAAAA' if is_noise else palette[i % len(palette)],
                    size=5 if is_noise else 6,
                    symbol='x' if is_noise else 'circle',
                    opacity=0.5 if is_noise else 0.75
                )
            ), row=1, col=2
        )

    fig.update_layout(
        title_text='Comparação Visual: K-Means vs DBSCAN',
        template='plotly_white',
        height=540,
        showlegend=True
    )
    fig.show()

In [46]:
# Tabela de métricas comparativas

metricas = {
    'Métrica': [
        'Algoritmo',
        'Nº de Clusters',
        'Nº de Outliers/Ruídos',
        '% de Ruído',
        'Silhouette Score',
        'Parâmetros Principais',
        'Define k a priori?',
        'Detecta outliers?',
        'Sensível à escala?',
        'Clusters não-esféricos?'
    ],
    'K-Means': [
        'K-Means',
        K_FINAL,
        '—',
        '—',
        f'{sil_final:.4f}',
        f'k={K_FINAL}',
        '✅ Sim',
        '❌ Não',
        '✅ Sim',
        '❌ Não (esféricos)'
    ],
    'DBSCAN': [
        'DBSCAN',
        n_clusters_db,
        n_noise_db,
        f'{noise_pct:.1f}%',
        f'{sil_db:.4f}' if not np.isnan(sil_db) else 'N/A',
        f'eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES}',
        '❌ Não',
        '✅ Sim',
        '✅ Sim',
        '✅ Sim (formas arbitrárias)'
    ]
}

df_comp = pd.DataFrame(metricas)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Métrica</b>', '<b>K-Means</b>', '<b>DBSCAN</b>'],
        fill_color=['#2C3E50', '#636EFA', '#00CC96'],
        font=dict(color='white', size=13),
        align='left',
        height=35
    ),
    cells=dict(
        values=[df_comp['Métrica'], df_comp['K-Means'], df_comp['DBSCAN']],
        fill_color=[['#F8F9FA'] * len(df_comp),
                    ['#EEF2FF'] * len(df_comp),
                    ['#E8FBF5'] * len(df_comp)],
        align='left',
        font=dict(size=12),
        height=30
    )
)])

fig.update_layout(
    title='📊 Comparação de Métricas e Características',
    template='plotly_white',
    height=450
)
fig.show()

In [47]:
# Distribuição de tamanho dos clusters — K-Means vs DBSCAN

km_counts = pd.Series(km_labels).value_counts().sort_index()
db_counts = pd.Series(db_labels).value_counts().sort_index()

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Tamanho dos Clusters — K-Means', 'Tamanho dos Clusters — DBSCAN')
)

palette = px.colors.qualitative.Bold

fig.add_trace(
    go.Bar(
        x=[f'Cluster {i}' for i in km_counts.index],
        y=km_counts.values,
        marker_color=[palette[i % len(palette)] for i in range(len(km_counts))],
        text=km_counts.values, textposition='outside',
        name='K-Means'
    ), row=1, col=1
)

db_labels_str = ['Ruído' if i == -1 else f'Cluster {i}' for i in db_counts.index]
db_colors = ['#AAAAAA' if i == -1 else palette[j % len(palette)]
             for j, i in enumerate(db_counts.index)]
fig.add_trace(
    go.Bar(
        x=db_labels_str,
        y=db_counts.values,
        marker_color=db_colors,
        text=db_counts.values, textposition='outside',
        name='DBSCAN'
    ), row=1, col=2
)

fig.update_layout(
    title_text='Distribuição do Tamanho dos Clusters',
    template='plotly_white',
    height=430,
    showlegend=False
)
fig.update_yaxes(title_text='Número de Pontos')
fig.show()

## 7. 📝 Conclusão

| | **K-Means** | **DBSCAN** |
|---|---|---|
| **Quando usar** | Clusters esféricos, tamanho similar, sem muitos outliers | Dados com formas arbitrárias, presença de ruído/outliers |
| **Vantagem principal** | Simples, escalável, interpretável | Não exige k a priori, robusto a outliers |
| **Desvantagem principal** | Exige definição de k, sensível a outliers | Sensível aos parâmetros eps e min_samples |
| **Silhouette Score** | Alto em blobs bem separados | Alto em dados com formas não-esféricas |

---

**Próximos passos com seus dados reais:**
1. Substitua o bloco `[SUBSTITUIR DADOS REAIS]` com seu DataFrame
2. Ajuste `FEATURE_COLS` para selecionar as colunas relevantes
3. Ajuste `K_RANGE`, `DBSCAN_EPS` e `DBSCAN_MIN_SAMPLES` conforme os resultados
4. Use o k-Distance Plot para calibrar o `eps` do DBSCAN
